# CodeForge ASM Autoresearch — Kaggle

**GPU:** Accelerator > GPU T4 x2  
**Secrets:** Add `CLAUDE_CODE_OAUTH_TOKEN` e `HF_TOKEN` em Add-ons > Secrets  
**Data:** Add dataset `codeforge-asm.zip` em Data (painel direito)

## 1. Auth

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["CLAUDE_CODE_OAUTH_TOKEN"] = secrets.get_secret("CLAUDE_CODE_OAUTH_TOKEN")
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
os.environ.pop("ANTHROPIC_API_KEY", None)

print("CLAUDE token:", os.environ["CLAUDE_CODE_OAUTH_TOKEN"][:12] + "...")
print("HF token:", os.environ["HF_TOKEN"][:12] + "...")

## 2. Instalar Claude Code CLI

In [ ]:
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - && apt-get install -y nodejs -q
!npm install -g @anthropic-ai/claude-code --silent
!claude --version

## 3. Extrair projeto

In [ ]:
import os
import glob

# Encontra o zip no input
zips = glob.glob("/kaggle/input/**/codeforge-asm.zip", recursive=True)
if not zips:
    zips = glob.glob("/kaggle/input/**/*.zip", recursive=True)
print("Zip encontrado:", zips[0])

!unzip -q "{zips[0]}" -d /kaggle/working/
!ls /kaggle/working/codeforge-asm/

## 4. Instalar dependencias

In [ ]:
import os
%cd /kaggle/working/codeforge-asm

!apt-get install -y nasm binutils -q
!nasm --version && ld --version | head -1
!pip install -q -r requirements.txt

print("Done!")

## 5. Preparar workspace

In [ ]:
%cd /kaggle/working/codeforge-asm
!git config user.email "autoresearch@kaggle.local"
!git config user.name "AutoResearch Agent"
!python prepare.py

## 6. Teste rapido

Confirma que o pipeline funciona antes de ligar o agente.

In [ ]:
%cd /kaggle/working/codeforge-asm
!python train.py 2>&1 | grep -E "primary_metric|correct_rate|assembly_rate|training_seconds" || echo "Cheque run.log"

## 7. Agente autonomo

Metrica: **correct_rate** (maior = melhor) | Baseline esperado: `~0.10-0.20`

Cada experimento = ~30-60 min (5 iteracoes GRPO no T4 x2)

In [ ]:
%%bash
export PATH="/usr/local/bin:/usr/bin:/bin:$PATH"
export CLAUDE_CODE_OAUTH_TOKEN=$(python3 -c "from kaggle_secrets import UserSecretsClient; print(UserSecretsClient().get_secret('CLAUDE_CODE_OAUTH_TOKEN'))")
export HF_TOKEN=$(python3 -c "from kaggle_secrets import UserSecretsClient; print(UserSecretsClient().get_secret('HF_TOKEN'))")
cd /kaggle/working/codeforge-asm

claude --dangerously-skip-permissions -p "\
Hi! Have a look at program.md and let's kick off a new experiment. \
Do the setup first (create branch, read files, verify data, run prepare.py, init results.tsv), \
confirm, then start the autonomous experiment loop."

## 8. Resultados

In [ ]:
import pandas as pd

try:
    df = pd.read_csv("/kaggle/working/codeforge-asm/results.tsv", sep="\t")
    keeps = df[df["status"] == "keep"]
    print(f"{len(df)} experimentos | {len(keeps)} kept | melhor correct_rate: {keeps['correct_rate'].max():.6f}\n")
    print(df.to_string(index=False))
except FileNotFoundError:
    print("Rode o agente primeiro.")